# Visual Agents & Computer Use: From Perception to Autonomous Action

## A PhD-Level Deep Dive into Vision-Language Models that *Act* on Graphical Interfaces

---

**Paradigm Shift (2025-2026):** Vision-Language Models (VLMs) have evolved from passive *describers* to active *agents*. Instead of answering "What color is the button?", modern visual agents answer "Book me a flight to Tokyo" by perceiving GUI elements, planning multi-step actions, and executing them in a closed loop.

```
Traditional VQA:                              Visual Agent:
  Input:  image + "What color is the car?"      Input:  screenshot + "Book a flight to Tokyo"
  Output: "Red"                                  Output: click(340, 120) -> type("Tokyo") -> click(500, 120)
```

**Architecture at a Glance:**

```
                        +------------------+
                        |   Screenshot     |
                        +--------+---------+
                                 |
                        +--------v---------+
                        |  Vision Encoder  |    Patch embeddings via ViT
                        |  (SigLIP, etc.)  |    
                        +--------+---------+
                                 |
                        +--------v---------+
                        |    Projector     |    MLP / Cross-attention bridge
                        +--------+---------+    aligning visual -> text latent space
                                 |
  Task instruction ---->+--------v---------+---> Action: click(340, 120)
                        |       LLM       |---> Action: type("Tokyo")
  Action history ------>|    (Decoder)    |---> Action: scroll(down)
                        +--------+---------+
                                 |
                      Execute action on screen
                                 |
                      Take new screenshot --> Loop back
```

**What this notebook covers:**
1. **Visual Grounding** - mapping natural language to pixel coordinates and bounding boxes
2. **GUI Element Recognition** - detecting buttons, text fields, and menus from screenshots
3. **Action Vocabulary & Generation** - defining atomic actions and planning multi-step sequences
4. **Tool Calling with Vision** - closed-loop screenshot -> tool -> screenshot pipelines
5. **End-to-End Visual Agent** - autonomous UI navigation with state management
6. **Browser Automation Demo** - simulated agent that perceives, decides, and acts
7. **Evaluation Framework** - measuring success rate, efficiency, and robustness

**Models referenced:** Qwen2-VL / Qwen3-VL, GLM-4V, Claude Computer Use, CogAgent

**Prerequisites:** Transformers, backpropagation, basic RL. No prior visual agent experience assumed.

# 1) Environment Setup & Dependencies

In [ ]:
# Core dependencies for visual agent development
!pip install -q torch torchvision pillow matplotlib numpy

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as T
from torchvision import models
from PIL import Image, ImageDraw, ImageFont
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np
import json
import re
import time
import random
from dataclasses import dataclass, field
from typing import List, Tuple, Dict, Optional, Any
from enum import Enum, auto
from collections import deque
import warnings
warnings.filterwarnings("ignore")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
print(f"PyTorch: {torch.__version__}")

# Reproducibility
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

# 2) Visual Grounding: From Text Descriptions to Coordinates

## 2.1 The Grounding Problem

Visual grounding is the foundation of any visual agent: given an image and a textual reference (e.g., "the search button"), produce *spatial coordinates* - either a bounding box `[x_min, y_min, x_max, y_max]` or a click point `(x, y)`.

**Three output formats used by modern VLMs:**

| Format | Example Output | Use Case |
|--------|---------------|----------|
| **Normalized bbox** | `[0.20, 0.30, 0.80, 0.90]` | Element localization |
| **Pixel point** | `(340, 120)` | Click actions |
| **Structured JSON** | `{"element": "button", "box": [...], "label": "Search"}` | Full GUI parsing |

**Why normalization matters:** Models output coordinates in `[0, 1]` range relative to image dimensions. This makes predictions resolution-independent - the same model works on 1920x1080 and 640x480 screens without retraining.

```
Normalized:  (0.5, 0.5)   <-- center of any image
  | denormalize to 1920x1080
Pixel:       (960, 540)
  | denormalize to 640x480
Pixel:       (320, 240)
```

**Sample Input/Output:**
```
Input:  screenshot of a web form + "the email input field"
Output: {"box": [0.12, 0.34, 0.88, 0.40], "point": (0.50, 0.37), "label": "email_input"}
```

In [ ]:
# ============================================================
# Coordinate system utilities for visual grounding
# ============================================================
# Visual agents must convert between three coordinate spaces:
#   1. Normalized [0,1] - model output space (resolution-independent)
#   2. Pixel - absolute coordinates for action execution
#   3. Token - discretized grid indices used by some VLMs (e.g., Qwen2-VL uses 1000x1000)


@dataclass
class BoundingBox:
    """
    Axis-aligned bounding box in normalized [0, 1] coordinates.
    Convention: (x_min, y_min) = top-left, (x_max, y_max) = bottom-right.
    """

    x_min: float
    y_min: float
    x_max: float
    y_max: float

    def __post_init__(self):
        # Clamp all coordinates to valid range
        self.x_min = max(0.0, min(1.0, self.x_min))
        self.y_min = max(0.0, min(1.0, self.y_min))
        self.x_max = max(0.0, min(1.0, self.x_max))
        self.y_max = max(0.0, min(1.0, self.y_max))
        assert self.x_min <= self.x_max, f"Invalid box: x_min={self.x_min} > x_max={self.x_max}"
        assert self.y_min <= self.y_max, f"Invalid box: y_min={self.y_min} > y_max={self.y_max}"

    @property
    def center(self) -> Tuple[float, float]:
        """Return center point in normalized coords."""
        return ((self.x_min + self.x_max) / 2, (self.y_min + self.y_max) / 2)

    @property
    def area(self) -> float:
        """Normalized area in [0, 1]."""
        return (self.x_max - self.x_min) * (self.y_max - self.y_min)

    def to_pixels(self, width: int, height: int) -> Tuple[int, int, int, int]:
        """
        Convert normalized box to absolute pixel coordinates.
        # (4,) normalized -> (4,) pixel integers
        """
        return (
            int(self.x_min * width),
            int(self.y_min * height),
            int(self.x_max * width),
            int(self.y_max * height),
        )

    def to_token_grid(self, grid_size: int = 1000) -> Tuple[int, int, int, int]:
        """
        Convert to discretized token coordinates (Qwen2-VL convention).
        Grid of grid_size x grid_size maps continuous [0,1] to discrete {0,...,grid_size-1}.
        # (4,) normalized -> (4,) token grid integers
        """
        return (
            int(self.x_min * grid_size),
            int(self.y_min * grid_size),
            int(self.x_max * grid_size),
            int(self.y_max * grid_size),
        )

    def iou(self, other: "BoundingBox") -> float:
        """
        Intersection over Union - the standard grounding metric.
        IoU >= 0.5 is the conventional threshold for 'correct' grounding.
        """
        inter_x_min = max(self.x_min, other.x_min)
        inter_y_min = max(self.y_min, other.y_min)
        inter_x_max = min(self.x_max, other.x_max)
        inter_y_max = min(self.y_max, other.y_max)

        if inter_x_min >= inter_x_max or inter_y_min >= inter_y_max:
            return 0.0

        inter_area = (inter_x_max - inter_x_min) * (inter_y_max - inter_y_min)
        union_area = self.area + other.area - inter_area
        return inter_area / union_area if union_area > 0 else 0.0


# Demonstrate coordinate conversions
box = BoundingBox(0.12, 0.34, 0.88, 0.40)
print(f"Normalized box:   ({box.x_min}, {box.y_min}, {box.x_max}, {box.y_max})")
print(f"Center (norm):    {box.center}")
print(f"Pixel (1920x1080):{box.to_pixels(1920, 1080)}")
print(f"Pixel (640x480):  {box.to_pixels(640, 480)}")
print(f"Token grid (1000):{box.to_token_grid(1000)}")
print(f"Area:             {box.area:.4f}")

# IoU demonstration
box_a = BoundingBox(0.1, 0.1, 0.5, 0.5)
box_b = BoundingBox(0.3, 0.3, 0.7, 0.7)
box_c = BoundingBox(0.8, 0.8, 1.0, 1.0)
print(f"\nIoU(A, B) overlapping:     {box_a.iou(box_b):.4f}")
print(f"IoU(A, C) non-overlapping: {box_a.iou(box_c):.4f}")

## 2.2 Grounding Output Parsers

Different VLMs encode spatial information in different textual formats. A robust visual agent needs parsers for each convention.

| Model | Format | Example |
|-------|--------|---------|
| Qwen2-VL | `<box>(x1, y1),(x2, y2)</box>` on 1000x1000 grid | `<box>(120, 340),(880, 400)</box>` |
| CogAgent | `[[x1, y1, x2, y2]]` normalized x1000 | `[[120, 340, 880, 400]]` |
| GLM-4V | `<ref>element</ref><box>[[x1,y1,x2,y2]]</box>` | `<ref>button</ref><box>[[120,340,880,400]]</box>` |

Parsing model output is deceptively tricky in practice - models hallucinate malformed coordinates, produce out-of-range values, or mix formats mid-response. The parsers below handle these edge cases.

In [ ]:
# ============================================================
# Robust parsers for VLM grounding outputs
# ============================================================
# Each parser extracts BoundingBox objects from model-specific text formats.
# All parsers normalize to [0, 1] range regardless of input convention.


class GroundingParser:
    """
    Unified parser that auto-detects and extracts bounding boxes from
    diverse VLM output formats. Handles malformed outputs gracefully.
    """

    # Pattern: <box>(x1, y1),(x2, y2)</box> - Qwen2-VL style, coords in [0, 1000]
    QWEN_PATTERN = re.compile(
        r"<box>\s*\(?(\d+(?:\.\d+)?),\s*(\d+(?:\.\d+)?)\)?\s*,?\s*"
        r"\(?(\d+(?:\.\d+)?),\s*(\d+(?:\.\d+)?)\)?\s*</box>"
    )

    # Pattern: [[x1, y1, x2, y2]] - CogAgent / GLM style, coords in [0, 1000]
    BRACKET_PATTERN = re.compile(
        r"\[\[\s*(\d+(?:\.\d+)?),\s*(\d+(?:\.\d+)?),"
        r"\s*(\d+(?:\.\d+)?),\s*(\d+(?:\.\d+)?)\s*\]\]"
    )

    # Pattern: click(x, y) or point(x, y) - action-oriented output
    CLICK_PATTERN = re.compile(
        r"(?:click|point|tap)\s*\(\s*(\d+(?:\.\d+)?),\s*(\d+(?:\.\d+)?)\s*\)"
    )

    # Pattern: <ref>label</ref><box>[[...]]</box> - GLM-4V ref+box
    REF_BOX_PATTERN = re.compile(
        r"<ref>([^<]+)</ref>\s*<box>\[\[(\d+(?:\.\d+)?),\s*(\d+(?:\.\d+)?),"
        r"\s*(\d+(?:\.\d+)?),\s*(\d+(?:\.\d+)?)\]\]</box>"
    )

    @staticmethod
    def _normalize_coord(val: float, grid_size: int = 1000) -> float:
        """Map from token grid to [0, 1]. Values already in [0,1] pass through."""
        if val > 1.0:
            return max(0.0, min(1.0, val / grid_size))
        return max(0.0, min(1.0, val))

    @classmethod
    def parse(cls, text: str, grid_size: int = 1000) -> List[Dict[str, Any]]:
        """
        Auto-detect format and extract all grounding results from model output.
        Returns list of dicts with 'box' (BoundingBox), optional 'label', and 'format'.
        """
        results = []

        # Try ref+box pattern first (most specific)
        for match in cls.REF_BOX_PATTERN.finditer(text):
            label, x1, y1, x2, y2 = match.groups()
            box = BoundingBox(
                cls._normalize_coord(float(x1), grid_size),
                cls._normalize_coord(float(y1), grid_size),
                cls._normalize_coord(float(x2), grid_size),
                cls._normalize_coord(float(y2), grid_size),
            )
            results.append({"box": box, "label": label.strip(), "format": "ref_box"})

        if results:
            return results

        # Try Qwen-style <box> tags
        for match in cls.QWEN_PATTERN.finditer(text):
            x1, y1, x2, y2 = [float(g) for g in match.groups()]
            box = BoundingBox(
                cls._normalize_coord(x1, grid_size),
                cls._normalize_coord(y1, grid_size),
                cls._normalize_coord(x2, grid_size),
                cls._normalize_coord(y2, grid_size),
            )
            results.append({"box": box, "label": None, "format": "qwen"})

        if results:
            return results

        # Try bracket-style [[x1,y1,x2,y2]]
        for match in cls.BRACKET_PATTERN.finditer(text):
            x1, y1, x2, y2 = [float(g) for g in match.groups()]
            box = BoundingBox(
                cls._normalize_coord(x1, grid_size),
                cls._normalize_coord(y1, grid_size),
                cls._normalize_coord(x2, grid_size),
                cls._normalize_coord(y2, grid_size),
            )
            results.append({"box": box, "label": None, "format": "bracket"})

        if results:
            return results

        # Try click-point pattern (convert to tiny box around point)
        for match in cls.CLICK_PATTERN.finditer(text):
            x, y = float(match.group(1)), float(match.group(2))
            nx = cls._normalize_coord(x, grid_size)
            ny = cls._normalize_coord(y, grid_size)
            epsilon = 0.01
            box = BoundingBox(
                max(0, nx - epsilon), max(0, ny - epsilon),
                min(1, nx + epsilon), min(1, ny + epsilon),
            )
            results.append({"box": box, "label": f"click@({x},{y})", "format": "click"})

        return results


# Test all parser formats
test_cases = [
    ("Qwen2-VL", '<box>(120, 340),(880, 400)</box>'),
    ("CogAgent", 'The search bar is at [[120, 340, 880, 400]]'),
    ("GLM-4V", '<ref>search button</ref><box>[[700, 50, 800, 90]]</box>'),
    ("Click action", 'I will click(500, 300) on the submit button'),
    ("Multi-box", '<box>(10,10),(200,200)</box> and <box>(300,300),(500,500)</box>'),
    ("Malformed", 'No coordinates here at all'),
]

parser = GroundingParser()
for name, text in test_cases:
    results = parser.parse(text)
    print(f"\n[{name}] Input: {text}")
    if results:
        for r in results:
            b = r["box"]
            print(f"  -> Box: ({b.x_min:.3f}, {b.y_min:.3f}, {b.x_max:.3f}, {b.y_max:.3f})"
                  f"  Label: {r['label']}  Format: {r['format']}")
    else:
        print("  -> No grounding found (graceful fallback)")

## 2.3 Synthetic GUI Generation for Training & Testing

Real GUI datasets (Rico, WebUI, ScreenSpot) are large and require special licensing. For experimentation and unit testing, we generate synthetic screenshots with known ground-truth coordinates. This is standard practice - the SeeClick and CogAgent papers both use synthetic data augmentation.

**Sample Input/Output:**
```
Input:  generate_gui_screenshot(elements=[Button("Search"), TextField("Email"), ...])
Output: (PIL.Image, List[GUIElement])  -- screenshot + ground-truth element positions
```

In [ ]:
# ============================================================
# Synthetic GUI screenshot generator
# ============================================================
# Generates realistic-looking GUI layouts with ground-truth bounding boxes.
# Used for development, testing, and evaluation without external data.


class GUIElementType(Enum):
    BUTTON = auto()
    TEXT_FIELD = auto()
    CHECKBOX = auto()
    DROPDOWN = auto()
    LINK = auto()
    LABEL = auto()
    ICON = auto()
    NAV_BAR = auto()
    SEARCH_BAR = auto()


@dataclass
class GUIElement:
    """A single interactive or visual element in a synthetic GUI."""

    element_type: GUIElementType
    label: str
    bbox: BoundingBox
    is_interactive: bool = True
    state: str = "default"


# Color palettes mimicking common UI frameworks
UI_COLORS = {
    GUIElementType.BUTTON: {"bg": "#4A90D9", "text": "#FFFFFF", "border": "#3A7BC8"},
    GUIElementType.TEXT_FIELD: {"bg": "#FFFFFF", "text": "#333333", "border": "#CCCCCC"},
    GUIElementType.CHECKBOX: {"bg": "#FFFFFF", "text": "#333333", "border": "#999999"},
    GUIElementType.DROPDOWN: {"bg": "#FFFFFF", "text": "#333333", "border": "#CCCCCC"},
    GUIElementType.LINK: {"bg": None, "text": "#1A73E8", "border": None},
    GUIElementType.LABEL: {"bg": None, "text": "#333333", "border": None},
    GUIElementType.NAV_BAR: {"bg": "#2C3E50", "text": "#FFFFFF", "border": "#1A252F"},
    GUIElementType.SEARCH_BAR: {"bg": "#FFFFFF", "text": "#999999", "border": "#DDDDDD"},
}


def generate_gui_screenshot(
    width: int = 1280,
    height: int = 720,
    num_elements: int = 12,
    seed: Optional[int] = None,
) -> Tuple[Image.Image, List[GUIElement]]:
    """
    Generate a synthetic GUI screenshot with diverse UI elements.
    Returns (image, elements) where each element has ground-truth bbox.
    Layout uses a grid-based placement to avoid overlap, mimicking real GUIs.
    """
    if seed is not None:
        random.seed(seed)

    img = Image.new("RGB", (width, height), "#F5F5F5")
    draw = ImageDraw.Draw(img)

    try:
        font = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf", 14)
        font_small = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf", 11)
        font_nav = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf", 14)
    except (IOError, OSError):
        font = ImageFont.load_default()
        font_small = font
        font_nav = font

    elements = []

    # Navigation bar at top
    nav_h = 48
    draw.rectangle([0, 0, width, nav_h], fill="#2C3E50")
    draw.text((20, 14), "MyApp", fill="#FFFFFF", font=font_nav)
    nav_items = ["Home", "Products", "About", "Contact"]
    nav_x = 150
    for item in nav_items:
        text_w = draw.textlength(item, font=font)
        bbox = BoundingBox(nav_x / width, 4 / height, (nav_x + text_w + 20) / width, (nav_h - 4) / height)
        draw.text((nav_x + 10, 14), item, fill="#ECF0F1", font=font)
        elements.append(GUIElement(GUIElementType.LINK, item, bbox))
        nav_x += text_w + 40

    # Search bar
    search_y, search_h = 70, 36
    search_x, search_w = width // 2 - 200, 400
    draw.rectangle(
        [search_x, search_y, search_x + search_w, search_y + search_h],
        fill="#FFFFFF", outline="#CCCCCC", width=1,
    )
    draw.text((search_x + 12, search_y + 9), "Search...", fill="#999999", font=font)
    elements.append(GUIElement(
        GUIElementType.SEARCH_BAR, "Search",
        BoundingBox(search_x / width, search_y / height,
                    (search_x + search_w) / width, (search_y + search_h) / height),
    ))

    # Content area with grid-based element placement
    content_top, content_bottom = 130, height - 60
    content_left, content_right = 40, width - 40

    element_templates = [
        (GUIElementType.BUTTON, ["Submit", "Save", "Cancel", "Login", "Sign Up", "Download"]),
        (GUIElementType.TEXT_FIELD, ["Username", "Email", "Password", "Phone"]),
        (GUIElementType.CHECKBOX, ["Remember me", "I agree to terms", "Subscribe"]),
        (GUIElementType.DROPDOWN, ["Country v", "Language v", "Category v"]),
        (GUIElementType.LABEL, ["Welcome back!", "Enter your details", "Settings"]),
    ]

    cols, rows = 3, 4
    cell_w = (content_right - content_left) / cols
    cell_h = (content_bottom - content_top) / rows

    placed = 0
    for row in range(rows):
        for col in range(cols):
            if placed >= num_elements:
                break

            etype, labels = random.choice(element_templates)
            label = random.choice(labels)
            colors = UI_COLORS[etype]

            cx = content_left + col * cell_w + random.uniform(10, cell_w * 0.2)
            cy = content_top + row * cell_h + random.uniform(5, cell_h * 0.2)

            if etype == GUIElementType.BUTTON:
                bw, bh = random.randint(90, 160), 36
                draw.rounded_rectangle([cx, cy, cx + bw, cy + bh], radius=4,
                                       fill=colors["bg"], outline=colors["border"])
                text_w = draw.textlength(label, font=font)
                draw.text((cx + (bw - text_w) / 2, cy + 9), label, fill=colors["text"], font=font)

            elif etype == GUIElementType.TEXT_FIELD:
                bw, bh = random.randint(180, 280), 36
                draw.rectangle([cx, cy, cx + bw, cy + bh], fill=colors["bg"], outline=colors["border"])
                draw.text((cx + 8, cy + 9), label, fill="#999999", font=font)

            elif etype == GUIElementType.CHECKBOX:
                bw_box, bh = 18, 18
                total_w = bw_box + 8 + draw.textlength(label, font=font_small) + 4
                draw.rectangle([cx, cy + 2, cx + bw_box, cy + 2 + bh],
                               fill=colors["bg"], outline=colors["border"])
                if random.random() > 0.5:
                    draw.text((cx + 3, cy + 1), "v", fill="#4A90D9", font=font)
                draw.text((cx + bw_box + 8, cy + 4), label, fill=colors["text"], font=font_small)
                bw = int(total_w)
                bh = 22

            elif etype == GUIElementType.DROPDOWN:
                bw, bh = random.randint(140, 200), 36
                draw.rectangle([cx, cy, cx + bw, cy + bh], fill=colors["bg"], outline=colors["border"])
                draw.text((cx + 8, cy + 9), label, fill=colors["text"], font=font)

            elif etype == GUIElementType.LABEL:
                bw = int(draw.textlength(label, font=font)) + 8
                bh = 24
                draw.text((cx, cy), label, fill=colors["text"], font=font)

            bbox = BoundingBox(cx / width, cy / height, (cx + bw) / width, (cy + bh) / height)
            elements.append(GUIElement(etype, label, bbox,
                                       is_interactive=(etype != GUIElementType.LABEL)))
            placed += 1

    # Footer
    draw.rectangle([0, height - 40, width, height], fill="#EEEEEE")
    draw.text((20, height - 28), "2025 MyApp Inc. | Privacy | Terms", fill="#999999", font=font_small)

    return img, elements


# Generate and display a synthetic screenshot with ground-truth boxes
screenshot, gui_elements = generate_gui_screenshot(seed=42)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 6))

ax1.imshow(screenshot)
ax1.set_title("Synthetic GUI Screenshot", fontsize=13)
ax1.axis("off")

ax2.imshow(screenshot)
element_colors = {
    GUIElementType.BUTTON: "red", GUIElementType.TEXT_FIELD: "blue",
    GUIElementType.CHECKBOX: "green", GUIElementType.DROPDOWN: "orange",
    GUIElementType.LINK: "cyan", GUIElementType.LABEL: "purple",
    GUIElementType.SEARCH_BAR: "magenta",
}

w, h = screenshot.size
for elem in gui_elements:
    px = elem.bbox.to_pixels(w, h)
    color = element_colors.get(elem.element_type, "yellow")
    rect = patches.Rectangle(
        (px[0], px[1]), px[2] - px[0], px[3] - px[1],
        linewidth=2, edgecolor=color, facecolor="none", linestyle="--"
    )
    ax2.add_patch(rect)
    ax2.text(px[0], px[1] - 4, f"{elem.element_type.name}: {elem.label}",
             fontsize=7, color=color, fontweight="bold",
             bbox=dict(boxstyle="round,pad=0.15", facecolor="white", alpha=0.8))

ax2.set_title("Ground-Truth Element Annotations", fontsize=13)
ax2.axis("off")
plt.tight_layout()
plt.show()

print(f"\nGenerated {len(gui_elements)} GUI elements:")
for e in gui_elements:
    b = e.bbox
    print(f"  {e.element_type.name:12s} | {e.label:20s} | "
          f"box=({b.x_min:.3f}, {b.y_min:.3f}, {b.x_max:.3f}, {b.y_max:.3f}) | "
          f"interactive={e.is_interactive}")

# 3) GUI Element Recognition

## 3.1 From Screenshots to Structured UI Trees

GUI element recognition transforms a raw screenshot into a structured representation of interactive elements - analogous to how a browser's DOM tree describes a webpage, but inferred purely from pixels.

**Why this is hard:**
- No DOM access for native apps, remote desktops, or game UIs
- Elements look radically different across OS themes and apps
- Overlapping elements (dropdowns, tooltips, modals)
- Variable resolution and DPI scaling

**The recognition pipeline:**
```
Screenshot -> Feature Extraction -> Region Proposal -> Classification -> Structured Output
     |              |                    |                |               |
  1280x720     CNN/ViT features    Candidate boxes    Button/Field/   JSON UI tree
  RGB image    (hidden_dim,)       from heuristics    Menu/etc.       with coords
```

We implement a lightweight recognition system using a pretrained vision backbone (ResNet-50 features) combined with a simple region proposal + classifier head. Production systems (CogAgent, Ferret-UI) use full VLM architectures, but the core principle is the same.

In [ ]:
# ============================================================
# GUI Element Recognizer: vision backbone + classification head
# ============================================================
# Architecture:
#   1. ResNet-50 backbone (pretrained, frozen) extracts spatial features
#   2. RoI (Region of Interest) pooling extracts features per candidate region
#   3. Classification head predicts element type + interactivity


class GUIElementRecognizer(nn.Module):
    """
    Lightweight GUI element recognition from screenshots.
    Uses pretrained ResNet-50 as a frozen feature extractor with a
    learnable classification head for element type prediction.
    """

    NUM_CLASSES = len(GUIElementType)

    def __init__(self, roi_output_size: int = 7, hidden_dim: int = 256):
        super().__init__()

        # Frozen ResNet-50 backbone extracts spatial feature maps
        backbone = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
        # Remove avg pool and FC, keep only conv layers
        # (batch_num, 3, H, W) -> (batch_num, 2048, H/32, W/32)
        self.features = nn.Sequential(*list(backbone.children())[:-2])
        for param in self.features.parameters():
            param.requires_grad = False

        self.roi_output_size = roi_output_size
        feat_dim = 2048

        # Classification head for element type
        # (batch_num, feat_dim * roi_h * roi_w) -> (batch_num, num_classes)
        self.classifier = nn.Sequential(
            nn.Linear(feat_dim * roi_output_size * roi_output_size, hidden_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(inplace=True),
            nn.Linear(hidden_dim // 2, self.NUM_CLASSES),
        )

        # Binary head: is this element interactive?
        # (batch_num, feat_dim * roi_h * roi_w) -> (batch_num, 1)
        self.interactivity_head = nn.Sequential(
            nn.Linear(feat_dim * roi_output_size * roi_output_size, hidden_dim // 2),
            nn.ReLU(inplace=True),
            nn.Linear(hidden_dim // 2, 1),
        )

    def extract_features(self, image: torch.Tensor) -> torch.Tensor:
        """
        Extract spatial feature map from screenshot.
        # (batch_num, 3, H, W) -> (batch_num, 2048, H/32, W/32)
        """
        with torch.no_grad():
            return self.features(image)

    def roi_pool(self, feature_map: torch.Tensor, boxes: torch.Tensor) -> torch.Tensor:
        """
        Extract fixed-size features for each candidate region via RoI Align.
        # feature_map: (batch_num, 2048, fH, fW)
        # boxes: (num_rois, 4) in normalized [0,1] coordinates
        # output: (num_rois, 2048, roi_size, roi_size)
        """
        from torchvision.ops import roi_align

        _, _, fh, fw = feature_map.shape

        # Scale normalized boxes to feature map coordinates
        # (num_rois, 4) -> (num_rois, 4) in feature-map pixel space
        scaled_boxes = boxes.clone()
        scaled_boxes[:, [0, 2]] *= fw
        scaled_boxes[:, [1, 3]] *= fh

        # roi_align expects [batch_index, x1, y1, x2, y2]
        # (num_rois, 4) -> (num_rois, 5)
        batch_indices = torch.zeros(scaled_boxes.size(0), 1, device=feature_map.device)
        rois = torch.cat([batch_indices, scaled_boxes], dim=1)

        return roi_align(feature_map, rois, output_size=self.roi_output_size, aligned=True)

    def forward(self, feature_map: torch.Tensor, boxes: torch.Tensor):
        """
        Classify candidate regions.
        # feature_map: (batch_num, 2048, fH, fW)
        # boxes: (num_rois, 4) normalized
        # -> class_logits: (num_rois, num_classes), interact_logits: (num_rois, 1)
        """
        # (num_rois, 2048, roi_size, roi_size)
        roi_features = self.roi_pool(feature_map, boxes)
        # (num_rois, 2048 * roi_size * roi_size)
        flat = roi_features.flatten(start_dim=1)
        return self.classifier(flat), self.interactivity_head(flat)


# Instantiate and test with synthetic data
recognizer = GUIElementRecognizer().to(DEVICE).eval()

# (1, 3, 720, 1280) - single screenshot
dummy_img = T.ToTensor()(screenshot).unsqueeze(0).to(DEVICE)

# (1, 3, 720, 1280) -> (1, 2048, 23, 40)
feat_map = recognizer.extract_features(dummy_img)
print(f"Input image shape:   {dummy_img.shape}")
print(f"Feature map shape:   {feat_map.shape}")
print(f"Spatial compression: {dummy_img.shape[-2]//feat_map.shape[-2]}x")

# Use ground-truth boxes as candidate regions
gt_boxes = torch.tensor(
    [[e.bbox.x_min, e.bbox.y_min, e.bbox.x_max, e.bbox.y_max] for e in gui_elements],
    device=DEVICE,
)

# (num_rois, num_classes), (num_rois, 1)
class_logits, interact_logits = recognizer(feat_map, gt_boxes)
print(f"\nCandidate regions:   {gt_boxes.shape[0]}")
print(f"Class logits shape:  {class_logits.shape}")
print(f"Interact logits:     {interact_logits.shape}")

predicted_classes = class_logits.argmax(dim=-1)
type_names = [t.name for t in GUIElementType]

print(f"\nPipeline verification (untrained model, predictions are random):")
for i, elem in enumerate(gui_elements[:6]):
    pred_name = type_names[predicted_classes[i].item()]
    print(f"  Element '{elem.label}': predicted={pred_name:12s} actual={elem.element_type.name}")

## 3.2 Heuristic Region Proposals for GUIs

Production visual agents use learned region proposal networks (RPNs), but GUI screenshots have strong structural priors that enable effective heuristic-based proposals. GUI elements tend to be rectangular, axis-aligned, and clustered in content regions.

**Key insight:** Unlike natural images where objects have arbitrary shapes, GUI elements are almost always axis-aligned rectangles with clear edges. This makes simple edge-based proposals surprisingly effective.

In [ ]:
# ============================================================
# Heuristic region proposal for GUI screenshots
# ============================================================
# Uses edge detection + contour analysis to propose candidate regions.
# Exploits the structural regularity of GUI layouts.


def propose_gui_regions(
    image: Image.Image,
    min_area_ratio: float = 0.0005,
    max_area_ratio: float = 0.25,
    edge_threshold: int = 30,
) -> List[BoundingBox]:
    """
    Generate candidate bounding boxes from a screenshot using edge-based heuristics.
    Strategy: grayscale -> gradient edges -> connected components -> filter by size -> NMS
    # image: PIL.Image (W, H, 3) -> List[BoundingBox] in normalized coords
    """
    img_array = np.array(image.convert("L"), dtype=np.float32)
    h, w = img_array.shape

    # Compute gradient magnitude via finite differences
    # (H, W) -> (H, W) gradient maps
    grad_x = np.zeros_like(img_array)
    grad_y = np.zeros_like(img_array)
    grad_x[:, 1:] = np.abs(img_array[:, 1:] - img_array[:, :-1])
    grad_y[1:, :] = np.abs(img_array[1:, :] - img_array[:-1, :])
    edges = np.sqrt(grad_x**2 + grad_y**2)

    binary = (edges > edge_threshold).astype(np.uint8)

    # Connected component labeling via BFS flood fill
    visited = np.zeros_like(binary, dtype=bool)
    proposals = []
    total_area = h * w

    for start_y in range(0, h, 4):
        for start_x in range(0, w, 4):
            if binary[start_y, start_x] == 0 or visited[start_y, start_x]:
                continue

            queue = deque([(start_y, start_x)])
            visited[start_y, start_x] = True
            min_r, max_r = start_y, start_y
            min_c, max_c = start_x, start_x
            component_size = 0

            while queue and component_size < 5000:
                r, c = queue.popleft()
                component_size += 1
                min_r, max_r = min(min_r, r), max(max_r, r)
                min_c, max_c = min(min_c, c), max(max_c, c)

                for dr, dc in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
                    nr, nc = r + dr, c + dc
                    if 0 <= nr < h and 0 <= nc < w and not visited[nr, nc] and binary[nr, nc]:
                        visited[nr, nc] = True
                        queue.append((nr, nc))

            box_area = (max_r - min_r) * (max_c - min_c)
            area_ratio = box_area / total_area

            if min_area_ratio <= area_ratio <= max_area_ratio and (max_r - min_r) > 5 and (max_c - min_c) > 10:
                proposals.append(BoundingBox(min_c / w, min_r / h, max_c / w, max_r / h))

    # Simple NMS: remove proposals with IoU > 0.5
    proposals.sort(key=lambda b: b.area, reverse=True)
    filtered = []
    for prop in proposals:
        if all(prop.iou(kept) < 0.5 for kept in filtered):
            filtered.append(prop)

    return filtered


proposals = propose_gui_regions(screenshot)
print(f"Proposed regions: {len(proposals)} (ground-truth: {len(gui_elements)})")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 6))
w_img, h_img = screenshot.size

ax1.imshow(screenshot)
ax1.set_title(f"Heuristic Proposals ({len(proposals)} regions)", fontsize=13)
for prop in proposals:
    px = prop.to_pixels(w_img, h_img)
    rect = patches.Rectangle(
        (px[0], px[1]), px[2] - px[0], px[3] - px[1],
        linewidth=1.5, edgecolor="lime", facecolor="none", alpha=0.7
    )
    ax1.add_patch(rect)
ax1.axis("off")

# Compute recall at IoU > 0.3
ax2.imshow(screenshot)
ax2.set_title("Proposal-GT Matching (IoU > 0.3)", fontsize=13)
matched = 0
for elem in gui_elements:
    best_iou = max((elem.bbox.iou(p) for p in proposals), default=0)
    px = elem.bbox.to_pixels(w_img, h_img)
    color = "lime" if best_iou > 0.3 else "red"
    rect = patches.Rectangle(
        (px[0], px[1]), px[2] - px[0], px[3] - px[1],
        linewidth=2, edgecolor=color, facecolor="none"
    )
    ax2.add_patch(rect)
    ax2.text(px[0], px[1] - 3, f"IoU={best_iou:.2f}", fontsize=7, color=color,
             bbox=dict(boxstyle="round,pad=0.1", facecolor="white", alpha=0.8))
    matched += 1 if best_iou > 0.3 else 0

ax2.axis("off")
plt.tight_layout()
plt.show()

recall = matched / len(gui_elements) if gui_elements else 0
print(f"\nProposal recall @ IoU>0.3: {matched}/{len(gui_elements)} = {recall:.1%}")

# 4) Action Vocabulary & Generation

## 4.1 Atomic Actions

Every visual agent needs a well-defined **action vocabulary** - the set of primitive operations it can execute on a screen. This is analogous to how RL agents have a discrete action space, but visual agents operate on pixel coordinates rather than game states.

**Standard action vocabulary (used by Claude Computer Use, CogAgent, OS-Atlas):**

| Action | Parameters | Semantics |
|--------|-----------|-----------|
| `click(x, y)` | pixel coordinates | Left-click at position |
| `double_click(x, y)` | pixel coordinates | Double-click |
| `right_click(x, y)` | pixel coordinates | Right-click (context menu) |
| `type(text)` | string | Type text into focused element |
| `press(key)` | key name | Press a keyboard key |
| `scroll(direction, amount)` | up/down/left/right, pixels | Scroll the view |
| `drag(x1, y1, x2, y2)` | start and end coords | Drag from point to point |
| `wait(seconds)` | duration | Pause execution |
| `screenshot()` | none | Capture current screen state |

**Design decision -- why atomic actions?** Coarser actions (e.g., `fill_form(data)`) would be easier for the model but fail on novel UIs. Atomic actions generalize to any interface because they mirror raw input events. The model learns to *compose* complex behaviors from primitives.

**Sample Input/Output:**
```
Input:  "Send an email to alice@example.com saying 'Hello'"
Output: [click(100, 50), click(300, 150), type("alice@example.com"),
         press("Tab"), type("Hello"), click(120, 500)]
```

In [ ]:
# ============================================================
# Action vocabulary: typed, validated, serializable atomic actions
# ============================================================


class ActionType(Enum):
    CLICK = "click"
    DOUBLE_CLICK = "double_click"
    RIGHT_CLICK = "right_click"
    TYPE = "type"
    PRESS = "press"
    SCROLL = "scroll"
    DRAG = "drag"
    WAIT = "wait"
    SCREENSHOT = "screenshot"
    DONE = "done"
    FAIL = "fail"


@dataclass
class Action:
    """
    A single atomic action in the visual agent's vocabulary.
    Coordinates are in absolute pixel space (converted from model's
    normalized output at execution time).
    """

    action_type: ActionType
    x: Optional[int] = None
    y: Optional[int] = None
    text: Optional[str] = None
    key: Optional[str] = None
    direction: Optional[str] = None
    amount: Optional[int] = None
    x2: Optional[int] = None
    y2: Optional[int] = None
    duration: Optional[float] = None
    confidence: float = 1.0
    reasoning: str = ""

    def validate(self, screen_w: int, screen_h: int) -> List[str]:
        """Check action parameters are within valid ranges. Return list of errors."""
        errors = []
        at = self.action_type

        if at in (ActionType.CLICK, ActionType.DOUBLE_CLICK, ActionType.RIGHT_CLICK):
            if self.x is None or self.y is None:
                errors.append(f"{at.value} requires (x, y) coordinates")
            elif not (0 <= self.x <= screen_w and 0 <= self.y <= screen_h):
                errors.append(f"Coordinates ({self.x}, {self.y}) out of bounds")

        if at == ActionType.TYPE and not self.text:
            errors.append("type() requires non-empty text")
        if at == ActionType.PRESS and not self.key:
            errors.append("press() requires a key name")
        if at == ActionType.SCROLL:
            if self.direction not in {"up", "down", "left", "right"}:
                errors.append("scroll direction must be up/down/left/right")
        if at == ActionType.DRAG:
            if any(v is None for v in (self.x, self.y, self.x2, self.y2)):
                errors.append("drag() requires (x, y, x2, y2)")

        return errors

    def to_text(self) -> str:
        """Serialize to the text format VLMs produce."""
        at = self.action_type
        if at in (ActionType.CLICK, ActionType.DOUBLE_CLICK, ActionType.RIGHT_CLICK):
            return f"{at.value}({self.x}, {self.y})"
        elif at == ActionType.TYPE:
            escaped = self.text.replace('"', '\\"')
            return f'type("{escaped}")'
        elif at == ActionType.PRESS:
            return f'press("{self.key}")'
        elif at == ActionType.SCROLL:
            return f"scroll({self.direction}, {self.amount or 3})"
        elif at == ActionType.DRAG:
            return f"drag({self.x}, {self.y}, {self.x2}, {self.y2})"
        elif at == ActionType.WAIT:
            return f"wait({self.duration or 1.0})"
        elif at == ActionType.SCREENSHOT:
            return "screenshot()"
        elif at in (ActionType.DONE, ActionType.FAIL):
            return f'{at.value}("{self.text or ""}")'
        return f"{at.value}()"


class ActionParser:
    """
    Parse VLM text output into validated Action objects.
    Handles multiple actions per response and various formatting conventions.
    """

    PATTERNS = {
        ActionType.CLICK: re.compile(r"(?<!double_)(?<!right_)click\s*\(\s*(\d+)\s*,\s*(\d+)\s*\)"),
        ActionType.DOUBLE_CLICK: re.compile(r"double_click\s*\(\s*(\d+)\s*,\s*(\d+)\s*\)"),
        ActionType.RIGHT_CLICK: re.compile(r"right_click\s*\(\s*(\d+)\s*,\s*(\d+)\s*\)"),
        ActionType.TYPE: re.compile(r'type\s*\(\s*["\'](.+?)["\'\s]*\)', re.DOTALL),
        ActionType.PRESS: re.compile(r'press\s*\(\s*["\']([\w+]+)["\']\s*\)'),
        ActionType.SCROLL: re.compile(r"scroll\s*\(\s*(up|down|left|right)\s*(?:,\s*(\d+))?\s*\)"),
        ActionType.DRAG: re.compile(r"drag\s*\(\s*(\d+)\s*,\s*(\d+)\s*,\s*(\d+)\s*,\s*(\d+)\s*\)"),
        ActionType.WAIT: re.compile(r"wait\s*\(\s*([\d.]+)\s*\)"),
        ActionType.SCREENSHOT: re.compile(r"screenshot\s*\(\s*\)"),
        ActionType.DONE: re.compile(r'done\s*\(\s*(?:["\'](.+?)["\'])?\s*\)'),
        ActionType.FAIL: re.compile(r'fail\s*\(\s*(?:["\'](.+?)["\'])?\s*\)'),
    }

    @classmethod
    def parse(cls, text: str) -> List[Action]:
        """Extract all actions from model output text, preserving order."""
        found = []
        for action_type, pattern in cls.PATTERNS.items():
            for match in pattern.finditer(text):
                action = cls._build_action(action_type, match)
                if action:
                    found.append((match.start(), action))
        found.sort(key=lambda x: x[0])
        return [action for _, action in found]

    @classmethod
    def _build_action(cls, action_type: ActionType, match: re.Match) -> Optional[Action]:
        """Construct an Action from a regex match."""
        groups = match.groups()
        if action_type in (ActionType.CLICK, ActionType.DOUBLE_CLICK, ActionType.RIGHT_CLICK):
            return Action(action_type, x=int(groups[0]), y=int(groups[1]))
        elif action_type == ActionType.TYPE:
            return Action(action_type, text=groups[0])
        elif action_type == ActionType.PRESS:
            return Action(action_type, key=groups[0])
        elif action_type == ActionType.SCROLL:
            return Action(action_type, direction=groups[0],
                          amount=int(groups[1]) if groups[1] else 3)
        elif action_type == ActionType.DRAG:
            return Action(action_type, x=int(groups[0]), y=int(groups[1]),
                          x2=int(groups[2]), y2=int(groups[3]))
        elif action_type == ActionType.WAIT:
            return Action(action_type, duration=float(groups[0]))
        elif action_type == ActionType.SCREENSHOT:
            return Action(action_type)
        elif action_type in (ActionType.DONE, ActionType.FAIL):
            return Action(action_type, text=groups[0] if groups[0] else "")
        return None


# Test the action parser with various model outputs
test_outputs = [
    'I need to click(340, 120) on the search bar.',
    'First click(100, 50) to open compose. Then type("alice@example.com"). '
    'Next press("Tab") and type("Hello!"). Finally click(120, 500) to send.',
    'scroll(down, 5) to see more, then wait(2.0) for loading.',
    'The task is complete. done("Email sent successfully")',
    'I should look at the screen but there is nothing to do.',
]

parser = ActionParser()
for output in test_outputs:
    actions = parser.parse(output)
    print(f"\nInput:  {output[:80]}{'...' if len(output) > 80 else ''}")
    print(f"Parsed: {len(actions)} action(s)")
    for a in actions:
        errors = a.validate(1280, 720)
        status = "ok" if not errors else f"ERR: {errors[0]}"
        print(f"  {a.to_text():40s} [{status}]")

## 4.2 Multi-Step Action Planning (ReAct Pattern)

A single user instruction ("Book a flight to Tokyo") requires 5-15 atomic actions. The **action planner** decomposes high-level goals into executable action sequences, maintaining a plan that can be revised based on observed screen states.

**Planning strategies in current systems:**

| Strategy | Description | Used By |
|----------|-------------|---------|
| **One-shot** | Model outputs entire action sequence at once | Early CogAgent |
| **ReAct-style** | Interleave thought -> action -> observation | Claude Computer Use |
| **Hierarchical** | High-level plan -> sub-plans -> atomic actions | OS-Atlas |

We implement ReAct-style planning because it allows **error recovery** - if an action produces an unexpected screen state, the agent can re-plan from the new observation rather than blindly executing a stale plan.

In [ ]:
# ============================================================
# ReAct-style action planner with state tracking
# ============================================================
# Maintains a thought-action-observation loop:
#   Thought:      "I need to click the search bar to enter my query"
#   Action:       click(640, 85)
#   Observation:  [new screenshot showing cursor in search bar]
#   Thought:      "Good, the search bar is focused. Now I type the query"
#   ...


@dataclass
class PlanStep:
    """One step in a ReAct-style plan."""
    thought: str
    action: Action
    observation: Optional[str] = None
    success: Optional[bool] = None


@dataclass
class ActionPlan:
    """
    A dynamic plan that tracks execution history and supports replanning.
    Plans are hypotheses validated against actual screen observations.
    """

    goal: str
    steps: List[PlanStep] = field(default_factory=list)
    current_step_idx: int = 0
    max_steps: int = 30
    status: str = "in_progress"

    @property
    def executed_steps(self) -> List[PlanStep]:
        return self.steps[: self.current_step_idx]

    @property
    def is_complete(self) -> bool:
        return self.status in ("success", "failure", "max_steps_reached")

    def add_step(self, thought: str, action: Action):
        self.steps.append(PlanStep(thought=thought, action=action))

    def record_observation(self, observation: str, success: bool):
        """Record the result of executing the current step."""
        if self.current_step_idx < len(self.steps):
            step = self.steps[self.current_step_idx]
            step.observation = observation
            step.success = success
            self.current_step_idx += 1
        if self.current_step_idx >= self.max_steps:
            self.status = "max_steps_reached"

    def replan_from_current(self, new_steps: List[PlanStep]):
        """Discard remaining steps and replace with new plan."""
        self.steps = self.steps[: self.current_step_idx] + new_steps

    def get_history_context(self, max_recent: int = 5) -> str:
        """Format recent execution history as context for the VLM."""
        recent = self.executed_steps[-max_recent:]
        lines = [f"Goal: {self.goal}", f"Progress: {self.current_step_idx}/{len(self.steps)} steps", ""]
        for i, step in enumerate(recent):
            step_num = self.current_step_idx - len(recent) + i + 1
            lines.append(f"Step {step_num}:")
            lines.append(f"  Thought: {step.thought}")
            lines.append(f"  Action:  {step.action.to_text()}")
            if step.observation:
                lines.append(f"  Result:  {step.observation}")
                lines.append(f"  Success: {step.success}")
            lines.append("")
        return "\n".join(lines)


# Demonstrate planning for "send an email"
plan = ActionPlan(goal="Send an email to alice@example.com saying 'Hello'")

simulated_execution = [
    ("I need to find and click the Compose button",
     Action(ActionType.CLICK, x=100, y=50, reasoning="Compose button"),
     "New email compose window opened", True),
    ("Compose window is open. Click the To field",
     Action(ActionType.CLICK, x=300, y=150, reasoning="To: field"),
     "Cursor is now in the To field", True),
    ("To field is focused. Type the email address",
     Action(ActionType.TYPE, text="alice@example.com"),
     "Email address entered", True),
    ("Address entered. Move to body",
     Action(ActionType.PRESS, key="Tab"),
     "Focus moved to email body", True),
    ("Body is focused. Type the message",
     Action(ActionType.TYPE, text="Hello"),
     "Message typed", True),
    ("Message ready. Click Send",
     Action(ActionType.CLICK, x=120, y=500, reasoning="Send button"),
     "Email sent confirmation shown", True),
    ("Email has been sent successfully",
     Action(ActionType.DONE, text="Email sent to alice@example.com"),
     "Task complete", True),
]

for thought, action, observation, success in simulated_execution:
    plan.add_step(thought, action)
    plan.record_observation(observation, success)

print(plan.get_history_context(max_recent=10))
print(f"Plan status: {plan.status}")

## 4.3 Error Recovery: What Happens When Actions Fail

In real GUIs, actions fail constantly - buttons move, pages load slowly, popups appear unexpectedly. A robust visual agent must detect failures and recover.

**Common failure modes and recovery strategies:**
```
Failure detected
    |-- Retry (same action, slight coord adjustment)
    |-- Dismiss obstacle (close popup/modal, then retry)
    |-- Re-observe (take fresh screenshot, re-locate target)
    |-- Replan (different approach entirely)
    +-- Abort (too many consecutive failures)
```

In [ ]:
# ============================================================
# Error recovery system for visual agents
# ============================================================
# Implements a hierarchical recovery strategy with escalation.


class RecoveryStrategy(Enum):
    RETRY = "retry"
    DISMISS_OBSTACLE = "dismiss_obstacle"
    REOBSERVE = "re_observe"
    REPLAN = "replan"
    ABORT = "abort"


@dataclass
class FailureRecord:
    """Records a single failure event with context."""
    step_idx: int
    action: Action
    expected: str
    observed: str
    recovery_used: RecoveryStrategy
    recovered: bool


class ErrorRecoveryEngine:
    """
    Manages failure detection and recovery for visual agent execution.
    Uses an escalation ladder: each consecutive failure on the same
    step escalates to a more aggressive recovery strategy.
    """

    ESCALATION = [
        RecoveryStrategy.RETRY,
        RecoveryStrategy.REOBSERVE,
        RecoveryStrategy.DISMISS_OBSTACLE,
        RecoveryStrategy.REPLAN,
        RecoveryStrategy.ABORT,
    ]

    def __init__(self, max_retries: int = 3):
        self.max_retries = max_retries
        self.failure_history: List[FailureRecord] = []
        self.consecutive_failures: int = 0

    def diagnose_failure(
        self, action: Action, expected_elements: List[str],
        observed_elements: List[str], screen_changed: bool,
    ) -> RecoveryStrategy:
        """Determine recovery strategy based on failure context."""
        unexpected = set(observed_elements) - set(expected_elements)

        strategy_idx = min(self.consecutive_failures, len(self.ESCALATION) - 1)
        base_strategy = self.ESCALATION[strategy_idx]

        # Click had no effect -> wrong coordinates -> re-observe
        if not screen_changed and action.action_type in (ActionType.CLICK, ActionType.DOUBLE_CLICK):
            return max(base_strategy, RecoveryStrategy.REOBSERVE,
                       key=lambda s: self.ESCALATION.index(s))

        # Popup keywords detected -> dismiss obstacle
        popup_keywords = {"cookie", "accept", "dismiss", "close", "allow", "notification"}
        if any(kw in elem.lower() for elem in unexpected for kw in popup_keywords):
            return RecoveryStrategy.DISMISS_OBSTACLE

        return base_strategy

    def record_failure(self, step_idx, action, expected, observed, strategy, recovered):
        self.failure_history.append(
            FailureRecord(step_idx, action, expected, observed, strategy, recovered)
        )
        self.consecutive_failures = 0 if recovered else self.consecutive_failures + 1

    def generate_recovery_action(self, strategy: RecoveryStrategy, original: Action) -> Optional[Action]:
        """Generate a concrete recovery action."""
        if strategy == RecoveryStrategy.RETRY:
            jitter = random.randint(-5, 5)
            return Action(original.action_type,
                          x=(original.x + jitter) if original.x else None,
                          y=(original.y + jitter) if original.y else None,
                          text=original.text, key=original.key,
                          reasoning=f"Retry: {original.reasoning}")
        elif strategy == RecoveryStrategy.DISMISS_OBSTACLE:
            return Action(ActionType.PRESS, key="Escape", reasoning="Dismiss popup/modal")
        elif strategy == RecoveryStrategy.REOBSERVE:
            return Action(ActionType.SCREENSHOT, reasoning="Re-observe screen state")
        return None

    def get_failure_summary(self) -> Dict[str, Any]:
        total = len(self.failure_history)
        if total == 0:
            return {"total_failures": 0, "recovery_rate": 1.0}
        recovered = sum(1 for f in self.failure_history if f.recovered)
        strategy_counts = {}
        for f in self.failure_history:
            s = f.recovery_used.value
            strategy_counts[s] = strategy_counts.get(s, 0) + 1
        return {"total_failures": total, "recovered": recovered,
                "recovery_rate": recovered / total, "strategies_used": strategy_counts}


# Simulate failure scenarios
recovery = ErrorRecoveryEngine()

scenarios = [
    {"name": "Click misalignment",
     "action": Action(ActionType.CLICK, x=340, y=120),
     "expected": ["search_bar"], "observed": ["search_bar"], "screen_changed": False},
    {"name": "Cookie popup blocks target",
     "action": Action(ActionType.CLICK, x=500, y=300),
     "expected": ["submit_button"],
     "observed": ["cookie_consent_dialog", "accept_button", "dismiss_link"],
     "screen_changed": True},
    {"name": "Page navigated away",
     "action": Action(ActionType.CLICK, x=200, y=400),
     "expected": ["email_compose"],
     "observed": ["login_page", "username_field"], "screen_changed": True},
]

print("Error Recovery Simulation")
print("=" * 60)
for scenario in scenarios:
    strategy = recovery.diagnose_failure(
        scenario["action"], scenario["expected"],
        scenario["observed"], scenario["screen_changed"],
    )
    recovery_action = recovery.generate_recovery_action(strategy, scenario["action"])
    recovered = strategy != RecoveryStrategy.ABORT

    recovery.record_failure(0, scenario["action"], str(scenario["expected"]),
                            str(scenario["observed"]), strategy, recovered)

    print(f"\n[{scenario['name']}]")
    print(f"  Original:  {scenario['action'].to_text()}")
    print(f"  Strategy:  {strategy.value}")
    if recovery_action:
        print(f"  Recovery:  {recovery_action.to_text()}")

print(f"\nSummary: {json.dumps(recovery.get_failure_summary(), indent=2)}")

# 5) Tool Calling with Vision: Closed-Loop Visual Reasoning

## 5.1 The Vision-Tool Loop Architecture

The breakthrough in 2025-era visual agents is **closed-loop tool calling with visual feedback**:

1. Receive a screenshot
2. Decide which tool/action to call
3. Execute the action
4. Receive a *new* screenshot showing the result
5. Reason about whether the action succeeded
6. Decide the next action

This is the core pattern used by Claude Computer Use, GLM-4V, and Qwen3-VL.

```
+----------------------------------------------------------------+
|                   CLOSED-LOOP AGENT CYCLE                      |
|                                                                |
|   Screenshot(t=0) --> VLM Reason --> Action Output             |
|                                         |                      |
|                                    Execute Action              |
|                                         |                      |
|   Screenshot(t=1) <-- Compare <-- New Screen State             |
|          |                                                     |
|          +---- Loop until done() or max_steps -------->        |
+----------------------------------------------------------------+
```

**Key design decisions:**
- **Why not plan everything from the first screenshot?** GUIs are stateful - clicking a button changes the entire screen. You cannot predict what appears until you click.
- **Why include screenshots in the loop?** Text-only state descriptions lose spatial information. The model needs to *see* that its click landed correctly.
- **Token cost:** Each screenshot costs ~1000-2000 tokens. The tradeoff is accuracy vs. cost.

In [ ]:
# ============================================================
# Visual Tool Registry: tools that operate on screen state
# ============================================================
# In a visual agent, "tools" return *images* (new screenshots),
# not just text. This is the key difference from standard LLM tool calling.


@dataclass
class ToolResult:
    """Result of executing a visual tool on screen state."""
    success: bool
    screenshot: Optional[Image.Image]
    text_feedback: str
    metadata: Dict[str, Any] = field(default_factory=dict)


class ScreenState:
    """
    Maintains the current screen state for the agent loop.
    In production, this wraps a browser driver, desktop controller, or
    Android adb connection. Here we simulate with synthetic GUIs.
    """

    def __init__(self, width: int = 1280, height: int = 720, seed: Optional[int] = None):
        self.width = width
        self.height = height
        self.current_screenshot: Optional[Image.Image] = None
        self.elements: List[GUIElement] = []
        self.focused_element: Optional[GUIElement] = None
        self.typed_text: Dict[str, str] = {}
        self.action_log: List[Tuple[Action, bool]] = []
        self._refresh_screen(seed)

    def _refresh_screen(self, seed=None):
        self.current_screenshot, self.elements = generate_gui_screenshot(
            self.width, self.height, seed=seed
        )

    def _find_element_at(self, x: int, y: int) -> Optional[GUIElement]:
        """Find the GUI element at pixel coordinates (x, y)."""
        nx, ny = x / self.width, y / self.height
        for elem in self.elements:
            b = elem.bbox
            if b.x_min <= nx <= b.x_max and b.y_min <= ny <= b.y_max:
                return elem
        return None

    def execute(self, action: Action) -> ToolResult:
        """Execute an action and return result with new screenshot."""
        errors = action.validate(self.width, self.height)
        if errors:
            return ToolResult(False, self.current_screenshot,
                              f"Invalid action: {'; '.join(errors)}")

        at = action.action_type

        if at in (ActionType.CLICK, ActionType.DOUBLE_CLICK, ActionType.RIGHT_CLICK):
            elem = self._find_element_at(action.x, action.y)
            if elem and elem.is_interactive:
                self.focused_element = elem
                self._draw_click_feedback(action.x, action.y, elem)
                feedback = f"Clicked {elem.element_type.name} '{elem.label}' at ({action.x}, {action.y})"
                self.action_log.append((action, True))
                return ToolResult(True, self.current_screenshot, feedback,
                                  {"element": elem.label, "type": elem.element_type.name})
            self.action_log.append((action, False))
            return ToolResult(False, self.current_screenshot,
                              f"No interactive element at ({action.x}, {action.y})")

        elif at == ActionType.TYPE:
            if self.focused_element and self.focused_element.element_type == GUIElementType.TEXT_FIELD:
                self.typed_text[self.focused_element.label] = action.text
                self._draw_typed_text(self.focused_element, action.text)
                self.action_log.append((action, True))
                return ToolResult(True, self.current_screenshot,
                                  f"Typed '{action.text}' into '{self.focused_element.label}'")
            self.action_log.append((action, False))
            return ToolResult(False, self.current_screenshot, "No text field focused")

        elif at == ActionType.PRESS:
            self.action_log.append((action, True))
            return ToolResult(True, self.current_screenshot, f"Pressed '{action.key}'")

        elif at == ActionType.SCROLL:
            self.action_log.append((action, True))
            return ToolResult(True, self.current_screenshot,
                              f"Scrolled {action.direction}")

        elif at == ActionType.SCREENSHOT:
            self.action_log.append((action, True))
            return ToolResult(True, self.current_screenshot, "Screenshot captured")

        elif at == ActionType.DONE:
            self.action_log.append((action, True))
            return ToolResult(True, self.current_screenshot, f"Task completed: {action.text}")

        self.action_log.append((action, True))
        return ToolResult(True, self.current_screenshot, f"Executed {at.value}")

    def _draw_click_feedback(self, x, y, elem):
        """Draw a visual click indicator on the screenshot."""
        draw = ImageDraw.Draw(self.current_screenshot)
        draw.line([(x - 10, y), (x + 10, y)], fill="red", width=2)
        draw.line([(x, y - 10), (x, y + 10)], fill="red", width=2)
        draw.ellipse([x - 6, y - 6, x + 6, y + 6], outline="red", width=2)
        px = elem.bbox.to_pixels(self.width, self.height)
        draw.rectangle([px[0], px[1], px[2], px[3]], outline="red", width=2)

    def _draw_typed_text(self, elem, text):
        """Overlay typed text on the text field."""
        draw = ImageDraw.Draw(self.current_screenshot)
        px = elem.bbox.to_pixels(self.width, self.height)
        try:
            font = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf", 13)
        except (IOError, OSError):
            font = ImageFont.load_default()
        draw.rectangle([px[0]+1, px[1]+1, px[2]-1, px[3]-1], fill="white")
        draw.text((px[0]+6, px[1]+9), text, fill="#333333", font=font)


# Create a screen and execute a sequence of actions
screen = ScreenState(seed=42)
print(f"Screen: {screen.width}x{screen.height}, {len(screen.elements)} elements, "
      f"{sum(1 for e in screen.elements if e.is_interactive)} interactive")

text_fields = [e for e in screen.elements if e.element_type == GUIElementType.TEXT_FIELD]
buttons = [e for e in screen.elements if e.element_type == GUIElementType.BUTTON]

if text_fields and buttons:
    tf = text_fields[0]
    btn = buttons[0]
    tf_center = tf.bbox.center
    btn_center = btn.bbox.center

    actions_seq = [
        Action(ActionType.CLICK,
               x=int(tf_center[0] * screen.width),
               y=int(tf_center[1] * screen.height)),
        Action(ActionType.TYPE, text="agent@example.com"),
        Action(ActionType.CLICK,
               x=int(btn_center[0] * screen.width),
               y=int(btn_center[1] * screen.height)),
    ]

    print(f"\nExecuting {len(actions_seq)} actions:")
    for action in actions_seq:
        result = screen.execute(action)
        status = "OK" if result.success else "FAIL"
        print(f"  {action.to_text():40s} [{status}] {result.text_feedback}")

    fig, ax = plt.subplots(1, 1, figsize=(12, 5))
    ax.imshow(screen.current_screenshot)
    ax.set_title("Screen After Action Execution (red = clicked elements)", fontsize=13)
    ax.axis("off")
    plt.tight_layout()
    plt.show()

# 6) End-to-End Visual Agent: The Complete System

## 6.1 Agent Architecture

Now we assemble all components into a complete visual agent integrating:
1. **Vision** - GUI element recognition from screenshots
2. **Planning** - ReAct-style thought-action-observation loop
3. **Execution** - Action parsing and screen interaction
4. **Recovery** - Error detection and strategy escalation

**The agent prompt template** tells the VLM what actions are available, what the screen shows, what has been done, and what the goal is. This is the critical interface between the orchestrator and the model.

```
System: You are a visual agent controlling a GUI.
Available actions: click(x,y), type(text), press(key), scroll(dir), done(msg), fail(msg)
Screen resolution: 1280x720

[Screenshot of current screen]

Goal: {user_instruction}
History: {previous_actions_and_observations}

Respond with:
Thought: <reasoning>
Action: <exactly one action>
```

In [ ]:
# ============================================================
# Complete Visual Agent: VLM-driven GUI automation
# ============================================================
# Full agent loop: observe -> think (VLM) -> act -> observe -> loop
# Uses a SimulatedVLM for demonstration; real VLMs plug in identically.


class SimulatedVLM:
    """
    Rule-based VLM simulator demonstrating the exact interface.
    Replace query() with a real VLM call for production use.
    """

    def __init__(self, screen_state: ScreenState):
        self.screen = screen_state
        self.step_counter = 0

    def query(self, screenshot, goal, history, available_elements):
        """
        Simulate VLM response. Real call would be:
            response = vlm.generate(images=[screenshot], prompt=...)
        """
        self.step_counter += 1
        goal_lower = goal.lower()

        if any(kw in goal_lower for kw in ["search", "find", "look up"]):
            target = next((e for e in available_elements
                           if e.element_type == GUIElementType.SEARCH_BAR), None)
            if target and self.step_counter == 1:
                cx, cy = target.bbox.center
                px, py = int(cx * self.screen.width), int(cy * self.screen.height)
                return (f"Thought: I see a search bar. I'll click on it first.\n"
                        f"Action: click({px}, {py})")

        if "type" in history.lower() or self.step_counter == 2:
            query_match = re.search(r'["\'](.*?)["\']', goal)
            query_text = query_match.group(1) if query_match else "query"
            return (f"Thought: Search bar is focused. I'll type my query.\n"
                    f'Action: type("{query_text}")')

        if self.step_counter >= 3:
            btn = next((e for e in available_elements
                        if e.element_type == GUIElementType.BUTTON), None)
            if btn:
                cx, cy = btn.bbox.center
                px, py = int(cx * self.screen.width), int(cy * self.screen.height)
                return (f"Thought: Query entered. Click submit.\n"
                        f"Action: click({px}, {py})")

        if self.step_counter >= 4:
            return 'Thought: Search submitted.\nAction: done("Search completed")'

        return "Thought: Let me observe the screen.\nAction: screenshot()"


class VisualAgent:
    """
    End-to-end visual agent orchestrator.
    Manages the observe-think-act loop, coordinates VLM, parser,
    screen executor, and error recovery.
    """

    def __init__(self, screen, vlm, max_steps=30, max_history=10):
        self.screen = screen
        self.vlm = vlm
        self.max_steps = max_steps
        self.max_history = max_history
        self.recovery = ErrorRecoveryEngine(max_retries=3)
        self.parser = ActionParser()
        self.execution_trace = []

    def run(self, goal: str) -> Dict[str, Any]:
        """Execute the full agent loop for a given goal."""
        plan = ActionPlan(goal=goal, max_steps=self.max_steps)
        start_time = time.time()
        step = 0

        print(f"\n{'='*60}")
        print(f"VISUAL AGENT EXECUTION")
        print(f"Goal: {goal}")
        print(f"{'='*60}")

        while step < self.max_steps and not plan.is_complete:
            step += 1
            print(f"\n--- Step {step} ---")

            # Query VLM with current state
            vlm_response = self.vlm.query(
                self.screen.current_screenshot, goal,
                plan.get_history_context(), self.screen.elements
            )
            print(f"VLM: {vlm_response}")

            # Parse action
            actions = self.parser.parse(vlm_response)
            if not actions:
                print("  [WARNING] No parseable action")
                plan.add_step("Parse failure", Action(ActionType.SCREENSHOT))
                plan.record_observation("No action parsed", False)
                continue

            action = actions[0]
            thought_match = re.search(r"Thought:\s*(.+?)(?:\n|$)", vlm_response)
            thought = thought_match.group(1) if thought_match else ""

            # Execute
            result = self.screen.execute(action)
            print(f"  Action:  {action.to_text()}")
            print(f"  Result:  {'OK' if result.success else 'FAIL'} {result.text_feedback}")

            plan.add_step(thought, action)
            plan.record_observation(result.text_feedback, result.success)
            self.execution_trace.append({
                "step": step, "thought": thought,
                "action": action.to_text(), "success": result.success,
                "feedback": result.text_feedback,
            })

            # Terminal actions
            if action.action_type == ActionType.DONE:
                plan.status = "success"
                break
            elif action.action_type == ActionType.FAIL:
                plan.status = "failure"
                break

            # Handle failures
            if not result.success:
                strategy = self.recovery.diagnose_failure(action, [], [], False)
                recovery_action = self.recovery.generate_recovery_action(strategy, action)
                if recovery_action:
                    print(f"  Recovery: {strategy.value} -> {recovery_action.to_text()}")
                    self.screen.execute(recovery_action)

        elapsed = time.time() - start_time
        summary = {
            "goal": goal, "status": plan.status, "total_steps": step,
            "successful_actions": sum(1 for t in self.execution_trace if t["success"]),
            "failed_actions": sum(1 for t in self.execution_trace if not t["success"]),
            "elapsed_seconds": round(elapsed, 3),
            "trace": self.execution_trace,
        }

        print(f"\n{'='*60}")
        print(f"COMPLETE: {plan.status} | Steps: {step} | Time: {elapsed:.2f}s")
        print(f"{'='*60}")
        return summary


# Run the agent on a search task
screen = ScreenState(seed=42)
vlm = SimulatedVLM(screen)
agent = VisualAgent(screen, vlm)
result = agent.run('Search for "Tokyo flights" using the search bar')

## 6.2 Connecting to Real VLMs

The `SimulatedVLM` above demonstrates the exact interface. To connect a real VLM, replace the `query` method. Below are reference patterns for Qwen2-VL (local) and API-based models (Claude Computer Use).

**Note:** Qwen2-VL-2B requires ~8GB GPU RAM. For Colab free tier, use the 2B variant or an API approach.

In [ ]:
# ============================================================
# Real VLM integration patterns (reference implementations)
# ============================================================
# These show the exact code to connect production VLMs.
# Uncomment and install dependencies to run with actual models.


class Qwen2VLAgent:
    """
    Integration pattern for Qwen2-VL.
    Outputs grounding coordinates on a 1000x1000 grid in <box> tags.

    Usage:
        pip install transformers accelerate
        agent = Qwen2VLAgent("Qwen/Qwen2-VL-2B-Instruct")
        agent.load()
        response = agent.query(screenshot, goal, history)
    """

    def __init__(self, model_name="Qwen/Qwen2-VL-2B-Instruct"):
        self.model_name = model_name

    def load(self):
        # from transformers import Qwen2VLForConditionalGeneration, AutoProcessor
        # self.processor = AutoProcessor.from_pretrained(self.model_name)
        # self.model = Qwen2VLForConditionalGeneration.from_pretrained(
        #     self.model_name, torch_dtype=torch.float16, device_map="auto")
        print(f"[Qwen2-VL] Would load {self.model_name}")

    def query(self, screenshot, goal, history):
        # messages = [{"role": "user", "content": [
        #     {"type": "image", "image": screenshot},
        #     {"type": "text", "text": f"Goal: {goal}\nHistory: {history}\n"
        #      "What action? Respond with Thought: and Action:"}
        # ]}]
        # text = self.processor.apply_chat_template(messages, add_generation_prompt=True)
        # inputs = self.processor(text=[text], images=[screenshot], return_tensors="pt")
        # output_ids = self.model.generate(**inputs, max_new_tokens=256)
        # return self.processor.batch_decode(output_ids, skip_special_tokens=True)[0]
        return "Thought: [Qwen2-VL]\nAction: screenshot()"


class APIBasedAgent:
    """
    Integration pattern for API-based VLMs (Claude Computer Use, GPT-4V).
    Claude's computer_use tool follows this exact pattern:
    send screenshot as base64 -> receive action in structured format.
    """

    def query(self, screenshot, goal, history):
        import base64, io
        buffer = io.BytesIO()
        screenshot.save(buffer, format="PNG")
        img_b64 = base64.b64encode(buffer.getvalue()).decode("utf-8")

        # response = client.messages.create(
        #     model="claude-sonnet-4-20250514",
        #     messages=[{"role": "user", "content": [
        #         {"type": "image", "source": {"type": "base64",
        #          "media_type": "image/png", "data": img_b64}},
        #         {"type": "text", "text": f"Goal: {goal}\n{history}\nNext action?"}
        #     ]}],
        #     tools=[{"type": "computer_20241022", "name": "computer",
        #             "display_width_px": 1280, "display_height_px": 720}]
        # )
        print(f"  [API] Would send {len(img_b64)} bytes to Claude")
        return "Thought: [API]\nAction: screenshot()"


print("VLM Integration Patterns:")
print("=" * 50)
print("\n1. Qwen2-VL (Local):")
print("   agent = Qwen2VLAgent('Qwen/Qwen2-VL-2B-Instruct')")
print("   agent.load(); response = agent.query(screenshot, goal, history)")
print("\n2. Claude Computer Use (API):")
print("   agent = APIBasedAgent()")
print("   response = agent.query(screenshot, goal, history)")
print("\nBoth return: 'Thought: ...\nAction: click(x, y)'")
print("The VisualAgent class works with either -- just swap the VLM.")

# 7) Demo: Simulated Browser Automation Agent

## 7.1 Multi-Page Navigation Simulation

This demo shows a complete agent navigating through a multi-page web application - simulating what happens when you tell Claude Computer Use to "book a flight."

The simulation includes multiple page states (home -> search results -> detail -> confirmation), page transitions triggered by clicking elements, and task completion verification. This is exactly how real browser automation agents work - the only difference is synthetic screens instead of a real browser.

In [ ]:
# ============================================================
# Multi-page browser simulation with stateful navigation
# ============================================================
# Simulates a travel booking website with 4 pages.
# Clicking certain elements triggers page transitions.


class WebPage(Enum):
    HOME = "home"
    SEARCH_RESULTS = "search_results"
    DETAIL = "detail"
    CONFIRMATION = "confirmation"


class BrowserSimulator:
    """Multi-page web app simulator for testing visual agents."""

    def __init__(self, width=1280, height=720):
        self.width = width
        self.height = height
        self.current_page = WebPage.HOME
        self.pages = {}
        self.transition_map = {}
        self._build_all_pages()

    def _get_fonts(self):
        try:
            return {
                "title": ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf", 20),
                "normal": ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf", 14),
                "small": ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf", 11),
            }
        except (IOError, OSError):
            f = ImageFont.load_default()
            return {"title": f, "normal": f, "small": f}

    def _create_page(self, title, bg="#F5F5F5"):
        img = Image.new("RGB", (self.width, self.height), bg)
        draw = ImageDraw.Draw(img)
        fonts = self._get_fonts()
        draw.rectangle([0, 0, self.width, 50], fill="#1A237E")
        draw.text((20, 12), "SkyBooker", fill="#FFFFFF", font=fonts["title"])
        draw.text((self.width - 200, 16), "Home | Flights | Help", fill="#B3B3FF", font=fonts["small"])
        draw.text((40, 70), title, fill="#333333", font=fonts["title"])
        return img, draw, fonts

    def _build_all_pages(self):
        self._build_home()
        self._build_results()
        self._build_detail()
        self._build_confirm()

    def _build_home(self):
        img, draw, fonts = self._create_page("Find Your Perfect Flight")
        elements = []

        # Destination search bar
        draw.rectangle([100, 130, 700, 170], fill="#FFFFFF", outline="#CCCCCC", width=2)
        draw.text((115, 140), "Where do you want to fly?", fill="#999999", font=fonts["normal"])
        elements.append(GUIElement(GUIElementType.SEARCH_BAR, "destination",
            BoundingBox(100/self.width, 130/self.height, 700/self.width, 170/self.height)))

        # Search button
        draw.rounded_rectangle([720, 130, 850, 170], radius=5, fill="#1A73E8")
        draw.text((745, 140), "Search Flights", fill="#FFFFFF", font=fonts["normal"])
        elements.append(GUIElement(GUIElementType.BUTTON, "Search Flights",
            BoundingBox(720/self.width, 130/self.height, 850/self.width, 170/self.height)))

        # Popular destinations
        y = 300
        draw.text((100, y - 20), "Popular Destinations:", fill="#333333", font=fonts["normal"])
        for dest in ["Tokyo", "Paris", "New York", "London"]:
            draw.rounded_rectangle([100, y, 250, y + 35], radius=4, fill="#E3F2FD")
            draw.text((115, y + 8), dest, fill="#1A73E8", font=fonts["normal"])
            elements.append(GUIElement(GUIElementType.BUTTON, dest,
                BoundingBox(100/self.width, y/self.height, 250/self.width, (y+35)/self.height)))
            self.transition_map[(WebPage.HOME, dest)] = WebPage.SEARCH_RESULTS
            y += 50

        self.pages[WebPage.HOME] = (img, elements)
        self.transition_map[(WebPage.HOME, "Search Flights")] = WebPage.SEARCH_RESULTS

    def _build_results(self):
        img, draw, fonts = self._create_page("Search Results: Flights to Tokyo")
        elements = []

        flights = [
            ("ANA NH101", "10:30 AM - 2:45 PM +1", "$890", "Direct"),
            ("JAL JL005", "11:00 AM - 3:20 PM +1", "$920", "Direct"),
            ("UA  UA837", "2:15 PM - 6:30 PM +1",  "$750", "1 Stop"),
        ]

        y = 120
        for airline, time_s, price, stops in flights:
            draw.rectangle([60, y, self.width - 60, y + 80], fill="#FFFFFF", outline="#E0E0E0")
            draw.text((80, y + 10), airline, fill="#333333", font=fonts["normal"])
            draw.text((80, y + 35), time_s, fill="#666666", font=fonts["small"])
            draw.text((500, y + 10), price, fill="#1A73E8", font=fonts["normal"])
            draw.text((500, y + 35), stops, fill="#666666", font=fonts["small"])

            bx = self.width - 200
            draw.rounded_rectangle([bx, y+20, bx+120, y+55], radius=4, fill="#4CAF50")
            draw.text((bx + 20, y + 28), "Book Now", fill="#FFFFFF", font=fonts["normal"])
            label = f"Book {airline.strip()[:3]}"
            elements.append(GUIElement(GUIElementType.BUTTON, label,
                BoundingBox(bx/self.width, (y+20)/self.height,
                            (bx+120)/self.width, (y+55)/self.height)))
            self.transition_map[(WebPage.SEARCH_RESULTS, label)] = WebPage.DETAIL
            y += 100

        self.pages[WebPage.SEARCH_RESULTS] = (img, elements)

    def _build_detail(self):
        img, draw, fonts = self._create_page("Complete Your Booking")
        elements = []

        fields = [("Full Name", 130), ("Email", 190), ("Phone", 250), ("Passport #", 310)]
        for label, y in fields:
            draw.text((100, y), label + ":", fill="#333333", font=fonts["normal"])
            draw.rectangle([250, y - 5, 600, y + 25], fill="#FFFFFF", outline="#CCCCCC")
            elements.append(GUIElement(GUIElementType.TEXT_FIELD, label,
                BoundingBox(250/self.width, (y-5)/self.height, 600/self.width, (y+25)/self.height)))

        draw.rounded_rectangle([250, 380, 430, 420], radius=5, fill="#4CAF50")
        draw.text((280, 390), "Confirm Booking", fill="#FFFFFF", font=fonts["normal"])
        elements.append(GUIElement(GUIElementType.BUTTON, "Confirm Booking",
            BoundingBox(250/self.width, 380/self.height, 430/self.width, 420/self.height)))
        self.transition_map[(WebPage.DETAIL, "Confirm Booking")] = WebPage.CONFIRMATION

        self.pages[WebPage.DETAIL] = (img, elements)

    def _build_confirm(self):
        img, draw, fonts = self._create_page("Booking Confirmed!")
        draw.text((100, 140), "Your flight has been booked!", fill="#4CAF50", font=fonts["title"])
        draw.text((100, 190), "Confirmation #: SKY-2025-78432", fill="#333333", font=fonts["normal"])
        draw.text((100, 220), "A confirmation email will be sent.", fill="#666666", font=fonts["normal"])
        self.pages[WebPage.CONFIRMATION] = (img, [])

    def get_screenshot(self):
        return self.pages[self.current_page]

    def click_element(self, label):
        key = (self.current_page, label)
        if key in self.transition_map:
            old = self.current_page
            self.current_page = self.transition_map[key]
            return True, f"Navigated {old.value} -> {self.current_page.value}"
        return False, f"No transition for '{label}' on {self.current_page.value}"


# Visualize all pages
browser = BrowserSimulator()
fig, axes = plt.subplots(2, 2, figsize=(18, 12))
pages = [WebPage.HOME, WebPage.SEARCH_RESULTS, WebPage.DETAIL, WebPage.CONFIRMATION]

for ax, page in zip(axes.flat, pages):
    browser.current_page = page
    img, elems = browser.get_screenshot()
    ax.imshow(img)
    ax.set_title(f"{page.value} ({len(elems)} elements)", fontsize=12)
    ax.axis("off")

plt.suptitle("Multi-Page Browser Simulation", fontsize=16, fontweight="bold")
plt.tight_layout()
plt.show()

# Simulate navigation flow
browser.current_page = WebPage.HOME
print("\nNavigation Flow:")
for elem, desc in [("Tokyo", "Select destination"), ("Book ANA", "Book first flight"),
                    ("Confirm Booking", "Confirm booking")]:
    ok, msg = browser.click_element(elem)
    print(f"  Click '{elem}': {'OK' if ok else 'FAIL'} {msg}")
print(f"Final page: {browser.current_page.value}")

In [ ]:
# ============================================================
# Full visual agent running on the browser simulator
# ============================================================


class BrowserScreenState(ScreenState):
    """Adapter: wraps BrowserSimulator as a ScreenState."""

    def __init__(self, browser):
        self.browser = browser
        self.width = browser.width
        self.height = browser.height
        self.action_log = []
        self.focused_element = None
        self.typed_text = {}
        self._update()

    def _update(self):
        self.current_screenshot, self.elements = self.browser.get_screenshot()

    def _find_element_at(self, x, y):
        nx, ny = x / self.width, y / self.height
        for elem in self.elements:
            b = elem.bbox
            if b.x_min <= nx <= b.x_max and b.y_min <= ny <= b.y_max:
                return elem
        return None

    def execute(self, action):
        at = action.action_type
        if at in (ActionType.CLICK, ActionType.DOUBLE_CLICK):
            elem = self._find_element_at(action.x, action.y)
            if elem:
                ok, msg = self.browser.click_element(elem.label)
                if ok:
                    self._update()
                    self.action_log.append((action, True))
                    return ToolResult(True, self.current_screenshot, msg,
                                      {"page": self.browser.current_page.value})
                self.focused_element = elem
                self.action_log.append((action, True))
                return ToolResult(True, self.current_screenshot,
                                  f"Clicked '{elem.label}' (no transition)")
            self.action_log.append((action, False))
            return ToolResult(False, self.current_screenshot,
                              f"No element at ({action.x}, {action.y})")
        return super().execute(action)


class BrowserVLM:
    """Smarter simulated VLM for multi-page navigation."""

    def __init__(self, browser):
        self.browser = browser
        self.step = 0

    def query(self, screenshot, goal, history, available_elements):
        self.step += 1
        page = self.browser.current_page

        if page == WebPage.HOME:
            tokyo_btn = next((e for e in available_elements if "Tokyo" in e.label), None)
            if tokyo_btn:
                cx, cy = tokyo_btn.bbox.center
                px, py = int(cx * self.browser.width), int(cy * self.browser.height)
                return (f"Thought: On home page. I see 'Tokyo' as popular destination. "
                        f"Clicking it directly.\nAction: click({px}, {py})")

        elif page == WebPage.SEARCH_RESULTS:
            book_btn = next((e for e in available_elements if "Book" in e.label), None)
            if book_btn:
                cx, cy = book_btn.bbox.center
                px, py = int(cx * self.browser.width), int(cy * self.browser.height)
                return (f"Thought: Search results shown. Booking first flight.\n"
                        f"Action: click({px}, {py})")

        elif page == WebPage.DETAIL:
            confirm = next((e for e in available_elements if "Confirm" in e.label), None)
            if confirm:
                cx, cy = confirm.bbox.center
                px, py = int(cx * self.browser.width), int(cy * self.browser.height)
                return (f"Thought: Booking form. Confirming booking.\n"
                        f"Action: click({px}, {py})")

        elif page == WebPage.CONFIRMATION:
            return ('Thought: Booking confirmed! Task complete.\n'
                    'Action: done("Flight to Tokyo booked. Confirmation #: SKY-2025-78432")')

        return "Thought: Observing.\nAction: screenshot()"


# Run the full browser automation demo
browser = BrowserSimulator()
browser.current_page = WebPage.HOME
screen = BrowserScreenState(browser)
vlm = BrowserVLM(browser)
agent = VisualAgent(screen, vlm)
result = agent.run("Book a flight to Tokyo")

# Visualize the execution trace
browser2 = BrowserSimulator()
pages_visited = [WebPage.HOME, WebPage.SEARCH_RESULTS, WebPage.DETAIL, WebPage.CONFIRMATION]
fig, axes = plt.subplots(1, 4, figsize=(22, 5))

for i, (ax, page) in enumerate(zip(axes, pages_visited)):
    browser2.current_page = page
    img, _ = browser2.get_screenshot()
    ax.imshow(img)
    trace = result["trace"]
    if i < len(trace):
        ax.set_title(f"Step {i+1}: {trace[i]['action'][:25]}...", fontsize=10)
    ax.axis("off")

plt.suptitle("Agent Execution Trace: Book a Flight to Tokyo", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

# 8) Evaluation: Measuring Visual Agent Performance

## 8.1 Metrics for Visual Agents

Evaluating visual agents differs fundamentally from VQA or object detection. We care about **task completion**, not just perception accuracy.

| Level | Metric | What It Measures |
|-------|--------|-----------------|
| **Task** | Success Rate (SR) | Did the agent complete the goal? |
| **Task** | Completion Rate (CR) | How far did it get? |
| **Efficiency** | Step Efficiency (SE) | `optimal_steps / actual_steps` |
| **Efficiency** | Action Precision (AP) | Fraction of useful actions |
| **Grounding** | Click Accuracy | Did clicks hit intended targets? |
| **Robustness** | Recovery Rate (RR) | Can it recover from errors? |
| **Safety** | Constraint Violation | Harmful actions (lower = better) |

A 100% success rate agent taking 50 steps for a 5-step task is poor. The evaluation framework captures these tradeoffs.

In [ ]:
# ============================================================
# Comprehensive evaluation framework for visual agents
# ============================================================


@dataclass
class TaskResult:
    """Result of running an agent on a single task."""
    task_id: str
    success: bool
    total_steps: int
    optimal_steps: int
    actions_taken: List[str]
    required_actions_completed: List[str]
    failed_actions: int
    recovery_events: int
    elapsed_seconds: float
    pages_visited: List[str]


class VisualAgentEvaluator:
    """
    Evaluation harness matching protocols from SeeClick, ScreenSpot,
    and OSWorld benchmarks.
    """

    def __init__(self):
        self.results: List[TaskResult] = []

    def add_result(self, result: TaskResult):
        self.results.append(result)

    def compute_metrics(self) -> Dict[str, float]:
        if not self.results:
            return {}

        n = len(self.results)
        success_rate = sum(1 for r in self.results if r.success) / n

        completion_rates = []
        for r in self.results:
            cr = len(r.required_actions_completed) / max(len(r.actions_taken), 1)
            completion_rates.append(min(1.0, cr))
        avg_completion = np.mean(completion_rates)

        step_effs = [min(1.0, r.optimal_steps / max(r.total_steps, 1)) for r in self.results]
        action_precs = [(r.total_steps - r.failed_actions) / max(r.total_steps, 1) for r in self.results]

        total_recoveries = sum(r.recovery_events for r in self.results)
        total_failures = sum(r.failed_actions for r in self.results)
        recovery_rate = total_recoveries / max(total_failures, 1) if total_failures > 0 else 1.0

        return {
            "num_tasks": n,
            "success_rate": round(success_rate, 4),
            "avg_completion_rate": round(avg_completion, 4),
            "avg_step_efficiency": round(np.mean(step_effs), 4),
            "avg_action_precision": round(np.mean(action_precs), 4),
            "recovery_rate": round(recovery_rate, 4),
            "avg_time_seconds": round(np.mean([r.elapsed_seconds for r in self.results]), 3),
        }

    def generate_report(self) -> str:
        metrics = self.compute_metrics()
        lines = ["=" * 60, "VISUAL AGENT EVALUATION REPORT", "=" * 60, ""]
        for key, val in metrics.items():
            if "rate" in key or "efficiency" in key or "precision" in key:
                lines.append(f"  {key:30s}: {val:.1%}")
            else:
                lines.append(f"  {key:30s}: {val}")
        lines.extend(["", "Per-task breakdown:", "-" * 60])
        for r in self.results:
            status = "PASS" if r.success else "FAIL"
            eff = min(1.0, r.optimal_steps / max(r.total_steps, 1))
            lines.append(f"  [{status}] {r.task_id:20s} | steps={r.total_steps}/{r.optimal_steps} "
                         f"(eff={eff:.0%}) | fails={r.failed_actions} | time={r.elapsed_seconds:.2f}s")
        return "\n".join(lines)


# Simulated evaluation results
evaluator = VisualAgentEvaluator()

tasks = [
    TaskResult("flight_booking", True, 5, 4, ["click"]*4+["done"],
               ["navigate", "select", "confirm"], 0, 0, 2.1, ["home", "results", "detail", "confirm"]),
    TaskResult("email_compose", True, 7, 6, ["click"]*2+["type"]*3+["click","done"],
               ["open", "fill_to", "fill_body", "send"], 1, 0, 3.5, ["inbox", "compose"]),
    TaskResult("form_fill", True, 9, 5, ["click","type"]*4+["click"],
               ["fill_name", "fill_email", "submit"], 2, 1, 4.2, ["form"]),
    TaskResult("web_search", True, 3, 3, ["click", "type", "click"],
               ["focus", "query", "submit"], 0, 0, 1.1, ["home", "results"]),
    TaskResult("login_flow", False, 12, 4, ["click"]*11+["fail"],
               ["enter_user", "enter_pass"], 5, 2, 8.7, ["login"]),
    TaskResult("settings_change", True, 6, 4, ["click"]*5+["done"],
               ["open_settings", "find_option", "toggle"], 1, 1, 2.8, ["home", "settings"]),
    TaskResult("file_upload", False, 15, 5, ["click"]*14+["fail"],
               ["open_dialog"], 8, 3, 12.1, ["home"]),
    TaskResult("calendar_event", True, 8, 6, ["click"]*2+["type"]*2+["click"]*3+["done"],
               ["open_cal", "new_event", "fill", "save"], 1, 0, 4.5, ["home", "calendar", "event"]),
]

for t in tasks:
    evaluator.add_result(t)

print(evaluator.generate_report())

In [ ]:
# ============================================================
# Visualization of evaluation results
# ============================================================

metrics = evaluator.compute_metrics()
results = evaluator.results

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Success rate gauge
ax1 = axes[0, 0]
sr = metrics["success_rate"]
color_sr = "#4CAF50" if sr > 0.7 else "#FF9800" if sr > 0.4 else "#F44336"
ax1.pie([sr, 1 - sr], colors=[color_sr, "#E0E0E0"], startangle=90, counterclock=False,
        wedgeprops={"linewidth": 2, "edgecolor": "white"})
ax1.text(0, 0, f"{sr:.0%}", ha="center", va="center", fontsize=28, fontweight="bold")
ax1.set_title("Task Success Rate", fontsize=13, pad=20)

# Per-task step efficiency
ax2 = axes[0, 1]
task_ids = [r.task_id.replace("_", "\n") for r in results]
efficiencies = [min(1.0, r.optimal_steps / max(r.total_steps, 1)) for r in results]
bar_colors = ["#4CAF50" if r.success else "#F44336" for r in results]
bars = ax2.barh(task_ids, efficiencies, color=bar_colors, edgecolor="white", linewidth=0.5)
ax2.set_xlim(0, 1.1)
ax2.axvline(x=1.0, color="gray", linestyle="--", alpha=0.5)
ax2.set_xlabel("Step Efficiency (optimal / actual)")
ax2.set_title("Per-Task Step Efficiency", fontsize=13)
for bar, eff in zip(bars, efficiencies):
    ax2.text(bar.get_width() + 0.02, bar.get_y() + bar.get_height()/2,
             f"{eff:.0%}", va="center", fontsize=9)

# Action breakdown
ax3 = axes[1, 0]
successful = [r.total_steps - r.failed_actions for r in results]
failed = [r.failed_actions for r in results]
x_pos = range(len(results))
ax3.bar(x_pos, successful, label="Successful", color="#4CAF50", alpha=0.8)
ax3.bar(x_pos, failed, bottom=successful, label="Failed", color="#F44336", alpha=0.8)
ax3.set_xticks(list(x_pos))
ax3.set_xticklabels([r.task_id.replace("_", "\n") for r in results], fontsize=8)
ax3.set_ylabel("Number of Actions")
ax3.set_title("Action Success/Failure per Task", fontsize=13)
ax3.legend(fontsize=9)

# Radar chart
ax4 = fig.add_subplot(2, 2, 4, polar=True)
radar_metrics = {
    "Success\nRate": metrics["success_rate"],
    "Step\nEfficiency": metrics["avg_step_efficiency"],
    "Action\nPrecision": metrics["avg_action_precision"],
    "Recovery\nRate": metrics["recovery_rate"],
    "Completion\nRate": metrics["avg_completion_rate"],
}
angles = np.linspace(0, 2 * np.pi, len(radar_metrics), endpoint=False).tolist()
values = list(radar_metrics.values())
angles += angles[:1]
values += values[:1]

ax4.set_theta_offset(np.pi / 2)
ax4.set_theta_direction(-1)
ax4.plot(angles, values, "o-", linewidth=2, color="#1A73E8")
ax4.fill(angles, values, alpha=0.15, color="#1A73E8")
ax4.set_xticks(angles[:-1])
ax4.set_xticklabels(list(radar_metrics.keys()), fontsize=9)
ax4.set_ylim(0, 1.0)
ax4.set_title("Aggregate Performance Radar", fontsize=13, pad=25)

plt.suptitle("Visual Agent Evaluation Dashboard", fontsize=16, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

# 9) Advanced Topics

## 9.1 Set-of-Mark (SoM) Prompting

Set-of-Mark (Yang et al., 2023) overlays numbered labels on detected UI elements before sending the screenshot to the VLM. The model outputs "click element 3" instead of "click(340, 120)", dramatically reducing coordinate hallucination.

```
Standard:  VLM sees raw screenshot -> outputs click(340, 120) -> might miss by 20px
SoM:       VLM sees labeled screenshot -> outputs "click [3]" -> we resolve [3] -> exact coords
```

**Resolution & Token Efficiency:**
| Resolution | Tokens (approx) | Strategy |
|-----------|-----------------|----------|
| 640x480 | ~400 | Navigation-level decisions |
| 1280x720 | ~1200 | Standard agent operation |
| 1920x1080 | ~2400 | Precise clicking tasks |

Dynamic resolution: low-res for navigation, high-res only for precise element interaction.

In [ ]:
# ============================================================
# Set-of-Mark (SoM) annotation for visual agents
# ============================================================
# Overlays numbered labels on elements so the VLM references by number
# instead of predicting raw coordinates.


def apply_set_of_mark(
    image: Image.Image, elements: List[GUIElement],
    label_size: int = 18, label_color: str = "#FF0000",
) -> Tuple[Image.Image, Dict[int, GUIElement]]:
    """
    Apply Set-of-Mark annotations to a GUI screenshot.
    # image: PIL.Image -> (annotated PIL.Image, Dict[int -> GUIElement])
    """
    annotated = image.copy()
    draw = ImageDraw.Draw(annotated)

    try:
        font = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf", label_size)
    except (IOError, OSError):
        font = ImageFont.load_default()

    label_map = {}
    w, h = image.size
    label_num = 1

    for elem in elements:
        if not elem.is_interactive:
            continue

        px = elem.bbox.to_pixels(w, h)
        label_x = px[0] - 2
        label_y = px[1] - label_size - 4
        label_text = str(label_num)
        text_w = draw.textlength(label_text, font=font)

        pill_w = max(text_w + 10, label_size + 6)
        draw.rounded_rectangle(
            [label_x, label_y, label_x + pill_w, label_y + label_size + 4],
            radius=10, fill=label_color, outline="#CC0000",
        )
        draw.text((label_x + (pill_w - text_w) / 2, label_y + 1),
                  label_text, fill="white", font=font)
        draw.rectangle(px, outline=label_color, width=2)

        label_map[label_num] = elem
        label_num += 1

    return annotated, label_map


# Generate SoM-annotated screenshot
screenshot, elements = generate_gui_screenshot(seed=42)
som_image, label_map = apply_set_of_mark(screenshot, elements)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 6))
ax1.imshow(screenshot)
ax1.set_title("Original Screenshot", fontsize=13)
ax1.axis("off")
ax2.imshow(som_image)
ax2.set_title("Set-of-Mark Annotated", fontsize=13)
ax2.axis("off")
plt.tight_layout()
plt.show()

print("\nSet-of-Mark Label Mapping:")
print("-" * 50)
for num, elem in label_map.items():
    center = elem.bbox.center
    print(f"  [{num:2d}] {elem.element_type.name:12s} '{elem.label}' at ({center[0]:.2f}, {center[1]:.2f})")

print(f"\nWith SoM, VLM outputs 'click [3]' instead of 'click(340, 120)'")
print("Orchestrator resolves [3] -> element center -> execute click")

## 9.2 Safety & Constraint Enforcement

Visual agents controlling real computers pose unique safety risks. An agent told to "clean up my desktop" should not delete important files.

**Key safety mechanisms:**
1. **Action allowlists** - restrict permitted actions per context
2. **Domain fencing** - restrict navigation to allowed URLs/apps
3. **Confirmation gates** - require human approval before irreversible actions
4. **Sensitive data filtering** - block typing of credit cards, SSNs, etc.
5. **Screen monitoring** - detect navigation to sensitive contexts

In [ ]:
# ============================================================
# Safety layer: action filtering and confirmation gates
# ============================================================


@dataclass
class SafetyPolicy:
    blocked_actions: List[ActionType] = field(default_factory=list)
    blocked_text_patterns: List[str] = field(default_factory=lambda: [
        r"\d{4}[\s-]?\d{4}[\s-]?\d{4}[\s-]?\d{4}",  # Credit card
        r"\b\d{3}-\d{2}-\d{4}\b",  # SSN
    ])
    confirmation_required: List[ActionType] = field(default_factory=lambda: [ActionType.DRAG])
    max_total_actions: int = 50


class SafetyGuard:
    """Intercepts and validates actions before execution."""

    def __init__(self, policy: SafetyPolicy):
        self.policy = policy
        self.action_count = 0
        self.blocked_log = []

    def check(self, action: Action) -> Tuple[bool, str]:
        self.action_count += 1

        if self.action_count > self.policy.max_total_actions:
            return False, f"Max actions ({self.policy.max_total_actions}) exceeded"

        if action.action_type in self.policy.blocked_actions:
            reason = f"'{action.action_type.value}' blocked by policy"
            self.blocked_log.append((action, reason))
            return False, reason

        if action.action_type == ActionType.TYPE and action.text:
            for pattern in self.policy.blocked_text_patterns:
                if re.search(pattern, action.text):
                    reason = f"Text matches blocked pattern"
                    self.blocked_log.append((action, reason))
                    return False, reason

        if action.action_type in self.policy.confirmation_required:
            return True, "REQUIRES_CONFIRMATION"

        return True, "allowed"


# Test safety guard
policy = SafetyPolicy(blocked_actions=[ActionType.RIGHT_CLICK])
guard = SafetyGuard(policy)

test_actions = [
    Action(ActionType.CLICK, x=100, y=200),
    Action(ActionType.TYPE, text="hello world"),
    Action(ActionType.TYPE, text="My card is 4111-1111-1111-1111"),
    Action(ActionType.TYPE, text="SSN: 123-45-6789"),
    Action(ActionType.RIGHT_CLICK, x=500, y=300),
    Action(ActionType.DRAG, x=10, y=10, x2=500, y2=500),
]

print("Safety Guard Results:")
print("=" * 70)
for action in test_actions:
    allowed, reason = guard.check(action)
    if allowed and reason == "allowed":
        status = "ALLOW"
    elif reason == "REQUIRES_CONFIRMATION":
        status = "CONFIRM"
    else:
        status = "BLOCK"
    print(f"  {status:8s} | {action.to_text():42s} | {reason}")

# 10) Summary & Key Takeaways

## What We Built

| Component | Section | Key Insight |
|-----------|---------|-------------|
| **Coordinate System** | 2.1 | Normalized [0,1] coordinates make models resolution-independent |
| **Grounding Parsers** | 2.2 | Different VLMs use different formats; robust parsing is essential |
| **GUI Generator** | 2.3 | Synthetic data enables rapid prototyping without real screenshots |
| **Element Recognizer** | 3.1 | Frozen ViT backbone + small classification head per region |
| **Region Proposals** | 3.2 | GUI regularity makes heuristic proposals surprisingly effective |
| **Action Vocabulary** | 4.1 | Atomic actions (click, type, scroll) generalize across all GUIs |
| **Action Planning** | 4.2 | ReAct-style (think->act->observe) enables dynamic replanning |
| **Error Recovery** | 4.3 | Escalation ladder: retry -> re-observe -> dismiss -> replan -> abort |
| **Tool Loop** | 5.1 | Closed-loop (screenshot -> VLM -> action -> new screenshot) is the core |
| **Full Agent** | 6.1 | Orchestrator integrating vision, planning, execution, and recovery |
| **Browser Demo** | 7.1 | Multi-page navigation with stateful transitions |
| **Evaluation** | 8.1 | Multi-level metrics: task success, step efficiency, action precision |
| **Set-of-Mark** | 9.1 | Label elements with numbers to reduce coordinate hallucination |
| **Safety** | 9.2 | Policy-based action filtering prevents harmful agent behavior |

## The Landscape (2025-2026)

| Model | Organization | Key Capability |
|-------|-------------|----------------|
| **Claude Computer Use** | Anthropic | API-based, tool_use with screenshots, highest reasoning |
| **Qwen2-VL / Qwen3-VL** | Alibaba | Open-weight, native grounding with `<box>` tags |
| **GLM-4V** | Zhipu AI | `<ref>` + `<box>` structured output, tool calling |
| **CogAgent** | Tsinghua/Zhipu | Specialized for GUI, high-res, `[[bbox]]` output |
| **OS-Atlas** | Shanghai AI Lab | Cross-platform (Windows/macOS/Linux/Android/Web) |

## What is Next

The field is moving toward: longer horizons (50+ step tasks), cross-application coordination, learning from demonstrations, and self-improving agents. The closed-loop perception architecture with ReAct planning in this notebook is the foundation all of these build upon.